# SMS message preparation and similarity
DS483 · Student 1 · Tasks 1 and 2

This notebook explores the UCI SMS Spam Collection, freezes a shared split for four later classifiers, and retrieves similar training messages using TF-IDF. It runs offline with the bundled `uci_sms.zip` alongside it. In Colab, upload that ZIP to the Files pane before running the cells. Run the cells in order. The result for teammates is `splits.csv`.

Source: T. Almeida and J. Hidalgo (2011), [SMS Spam Collection](https://doi.org/10.24432/C5CC84), UCI Machine Learning Repository, CC BY 4.0. The archive is the unmodified UCI download. AI assistance: ChatGPT generated the implementation and initial interpretation.

In [1]:
from pathlib import Path
import hashlib
import html
import re
import unicodedata
from zipfile import ZipFile

import numpy as np
import pandas as pd
import sklearn
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import StratifiedGroupKFold

with ZipFile(Path("uci_sms.zip")) as archive:
    raw = archive.read("SMSSpamCollection")
expected_hash = "7d039a24a6083ed9ef0f806ebad56bbb976e3aeb8de05669173bfdc4996c239d"
assert hashlib.sha256(raw).hexdigest() == expected_hash, "Unexpected source file"
records = []
for line_number, line in enumerate(raw.decode("utf-8").split("\n"), start=1):
    if line.strip():
        label, text = line.rstrip("\r").split("\t", 1)
        records.append((f"sms_{line_number:05d}", label, text))
sms = pd.DataFrame(records, columns=["message_id", "label", "text"])
assert sms.message_id.is_unique and set(sms.label) == {"ham", "spam"}
assert sms.text.str.strip().ne("").all()
counts = sms.label.value_counts()
print(pd.DataFrame({"messages": counts, "percent": (100 * counts / len(sms)).round(2)}).to_string())
print("Records:", len(sms), "| pandas:", pd.__version__, "| scikit-learn:", sklearn.__version__)

       messages  percent
label                   
ham        4827     86.6
spam        747     13.4
Records: 5574 | pandas: 2.2.3 | scikit-learn: 1.8.0


## Task 1 · Prepare the text

Keep the original text for the sequence and transformer experiments. For the lexical representation, normalize Unicode and apostrophes, decode HTML entities, convert to lowercase and replace URLs, digit sequences and currency symbols with category tokens. Keep words, including short SMS forms and contractions. Discard remaining punctuation in the lexical view and examine punctuation separately in the raw data. Retain stopwords and inflections because short replies and negation carry meaning. Normalizing numbers discards their exact values, and treating punctuation-only messages as empty loses distinctions among emoticons.

In [2]:
def clean_sms(text):
    text = unicodedata.normalize("NFKC", html.unescape(text)).casefold()
    text = text.replace("’", "'").replace("‘", "'")
    text = re.sub(r"(?:https?://|www\.)\S+", " urltoken ", text)
    text = re.sub(r"[£$€]", " moneytoken ", text)
    text = re.sub(r"\d+", " numtoken ", text)
    words = re.findall(r"[^\W\d_]+(?:'[^\W\d_]+)*", text)
    return " ".join(words)

sms["clean_text"] = sms.text.map(clean_sms)
print("Extra copies of identical raw messages:", sms.text.duplicated().sum())
print("Extra copies after cleaning:", sms.clean_text.duplicated().sum())
print("Rows empty after cleaning:", sms.clean_text.eq("").sum())
print("Clean-text groups containing both labels:", (sms.groupby("clean_text").label.nunique() > 1).sum())

Extra copies of identical raw messages: 403
Extra copies after cleaning: 477
Rows empty after cleaning: 2
Clean-text groups containing both labels: 0


## Freeze the split for all four classifiers

Seven approximately stratified folds allow two different folds for validation and test. The remaining five form training. Grouping by cleaned text keeps messages with the same lexical representation together, including punctuation-only messages. The source keeps repeated observations, so repeated messages can carry extra weight within a split. The check covers identical cleaned texts. Similar templates with changed wording may still occur across splits.

In [3]:
folds = StratifiedGroupKFold(n_splits=7, shuffle=True, random_state=483)
sms["split"] = "train"
for fold, (_, indices) in enumerate(folds.split(sms, sms.label, groups=sms.clean_text)):
    if fold < 2:
        sms.loc[indices, "split"] = ["test", "validation"][fold]

assert sms.groupby("clean_text").split.nunique().max() == 1
assert set(sms.split) == {"train", "validation", "test"}
split_table = pd.crosstab(sms.split, sms.label).reindex(["train", "validation", "test"])
split_table["total"] = split_table.sum(axis=1)
split_table["spam_pct"] = (100 * split_table.spam / split_table.total).round(2)
print(split_table.to_string())
sms[["message_id", "split", "label", "text", "clean_text"]].to_csv("splits.csv", index=False)
train = sms.loc[sms.split.eq("train")].reset_index(drop=True)

label        ham  spam  total  spam_pct
split                                  
train       3447   533   3980     13.39
validation   690   107    797     13.43
test         690   107    797     13.43


## Which features distinguish the training messages?

Describe the training subset so the group can make modeling decisions before seeing test performance. Percentages below are fractions of messages within each class. Character counts include spaces and punctuation.

In [4]:
signals = train.assign(
    characters=train.text.str.len(),
    has_digit=train.text.str.contains(r"\d"),
    has_exclamation=train.text.str.contains("!", regex=False),
    has_currency=train.text.str.contains(r"[£$€]"),
)
profile = signals.groupby("label").agg(
    median_chars=("characters", "median"),
    digit_pct=("has_digit", "mean"),
    exclamation_pct=("has_exclamation", "mean"),
    currency_pct=("has_currency", "mean"),
)
profile.iloc[:, 1:] = profile.iloc[:, 1:] * 100
print(profile.round(2).to_string())

       median_chars  digit_pct  exclamation_pct  currency_pct
label                                                        
ham            53.0      15.61            11.52          0.38
spam          148.0      93.81            50.28         34.33


## What changes in individual messages?

Select short training examples with identifiable cues. The original and cleaned text are printed side by side by ID.

In [5]:
samples = [
    ("ham", train.text.str.contains("!", regex=False)),
    ("spam", train.text.str.contains("£", regex=False)),
]
for label, cue in samples:
    row = train.loc[train.label.eq(label) & cue & train.text.str.len().between(20, 160)].iloc[0]
    print(row.message_id, "|", label)
    print("Original:", row.text)
    print("Cleaned: ", row.clean_text, "\n")

sms_00015 | ham
Original: I HAVE A DATE ON SUNDAY WITH WILL!!
Cleaned:  i have a date on sunday with will 

sms_00013 | spam
Original: URGENT! You have won a 1 week FREE membership in our £100,000 Prize Jackpot! Txt the word: CLAIM to No: 81010 T&C www.dbuk.net LCCLTD POBOX 4403LDNW1A7RW18
Cleaned:  urgent you have won a numtoken week free membership in our moneytoken numtoken numtoken prize jackpot txt the word claim to no numtoken t c urltoken lccltd pobox numtoken ldnw numtoken a numtoken rw numtoken 



### What the training messages show

The archive contains 4,827 ham and 747 spam messages (86.60% and 13.40%). On the training subset, the median spam message had 148 characters, versus 53 for ham. Digits occurred in 93.81% of spam and 15.61% of ham. Currency symbols appeared in 34.33% of spam and 0.38% of ham. Exclamation marks appeared in 50.28% and 11.52%, respectively. The all-caps Sunday-date example is a genuine message, so capitalization or punctuation alone would wrongly implicate some legitimate messages.

## Task 2 · Retrieve similar messages

Fit a unigram TF-IDF vocabulary to the training messages only. Here IDF is 1 + ln((1 + number of training messages) / (1 + number containing the term)). Within a message, multiply the term count by its IDF and normalize the vector to unit length. Cosine similarity then measures the overlap between weighted term vectors. It is zero for disjoint vocabularies. Texts with identical cleaned wording are excluded from each query’s candidates so that an exact copy cannot win.

In [6]:
vectorizer = TfidfVectorizer(tokenizer=str.split, token_pattern=None, lowercase=False, norm="l2")
vectors = vectorizer.fit_transform(train.clean_text)
terms = vectorizer.get_feature_names_out()
print("Training messages:", vectors.shape[0], "| terms:", vectors.shape[1])

Training messages: 3980 | terms: 6472


In [7]:
def describe_neighbor(query_index):
    scores = cosine_similarity(vectors[query_index], vectors).ravel()
    eligible = train.clean_text.ne(train.at[query_index, "clean_text"]).to_numpy()
    scores[~eligible] = -1
    neighbor = int(np.argmax(scores))
    contributions = vectors[query_index].multiply(vectors[neighbor]).tocoo()
    strongest = np.argsort(-contributions.data)[:5]
    shared = [terms[contributions.col[j]] for j in strongest]
    print("Query:", train.at[query_index, "message_id"], "|", train.at[query_index, "label"],
          "|", train.at[query_index, "text"])
    print("Match:", train.at[neighbor, "message_id"], "|", train.at[neighbor, "label"],
          "|", train.at[neighbor, "text"])
    print(f"Cosine: {scores[neighbor]:.4f} | highest shared contributions: {', '.join(shared)}\n")
    return query_index, neighbor, float(scores[neighbor]), shared

# Choose everyday and subscription query phrases before searching for each nearest neighbor.
results = []
for label, topic in [("ham", "catch the bus"), ("spam", "ringtone"), ("ham", "sorry, i")]:
    mask = train.label.eq(label) & train.text.str.lower().str.contains(topic, regex=False)
    query_index = int(train.index[mask][0])
    results.append(describe_neighbor(query_index))

Query: sms_00028 | ham | Did you catch the bus ? Are you frying an egg ? Did you make a tea? Are you eating your mom's left over dinner ? Do you feel my Love ?
Match: sms_05020 | ham | Babe ! What are you doing ? Where are you ? Who are you talking to ? Do you think of me ? Are you being a good boy? Are you missing me? Do you love me ?
Cosine: 0.3609 | highest shared contributions: you, are, do, love, a

Query: sms_00096 | spam | Your free ringtone is waiting to be collected. Simply text the password "MIX" to 85069 to verify. Get Usher and Britney. FML, PO Box 5249, MK17 92H. 450Ppw 16
Match: sms_00402 | spam | FREE RINGTONE text FIRST to 87131 for a poly or text GET to 87131 for a true tone! Help? 0845 2814032 16 after 1st free, tones are 3x£150pw to e£nd txt stop
Cosine: 0.3626 | highest shared contributions: numtoken, to, text, ringtone, free

Query: sms_00058 | ham | Sorry, I'll call later in meeting.
Match: sms_05063 | ham | Sorry, I'll call you  later. I am in meeting sir.
Cosine

### What the retrieved pairs show

The two meeting replies (sms_00058 and sms_05063) scored 0.8149. Both say the sender is in a meeting and will call later. The ringtone offers (sms_00096 and sms_00402) scored 0.3626. Shared terms include ringtone, free and text, and normalizing multiple different numbers increases their overlap. The two questions (sms_00028 and sms_05020) scored 0.3609 despite different topics. Their repeated you, are and do contributed to the match. Very similar scores can therefore reflect topical overlap or simply a shared conversational style. These illustrative pairs do not measure classification accuracy.

## Handoff for Tasks 3 to 5

Read the delivered CSV with `pd.read_csv("splits.csv", keep_default_na=False)`. Use its `split` column directly. `text` is the original message and `clean_text` is the lexical version. Fit any new text vocabulary on training only. Use validation to choose settings and evaluate final models on the same test IDs. Re-run this notebook from a folder containing the original ZIP to reproduce the handoff.